# Prepare Training Data for Mask R-CNN
This Jupyter notebook should be used to prepare data for training torchvision Mask R-CNN model. The mask annotations (for different instances of objects) are  used to create the training data in the following steps:
- Read the mask annotations (by the annotators) and parse them.  
- Crop the large images to smaller sub-images to keep the object sizes large enough for the model to be able to detect them. 
- Save the resulting sub-images and the annotations to disk to be read during the training. 

In [ ]:
# load required libraries
import os
from PIL import Image
from IPython.display import display
import numpy as np
import shutil
import cv2
import pandas as pd
from typing import List, Union, Dict, Final

### Data model - Reading annotated data (by annotators)
The data model class below returns masks as well.

In [ ]:
import sys
sys.path.append('../utils')
from json_parser import CellMaskDataset, optimize_crop, crop_and_block, show_sample

### Train and test sets
The mapping between the class IDs and the class names should have the same class names used in the annotations as the values. 

In [ ]:
# this folder will include all images (train or test)
# if an image is missing, it will get downloaded from the url specified in the annotation file
# the image names should be the same as the annotation file names
IMAGES_PATH = '/home/cellareye/Cellanome/Data/Images'

TRAIN_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/train'
TEST_ANNOTS_PATH = '/home/cellareye/Cellanome/Data/microscope-images-batch-1-091922-not-reviewed/test'

LABEL_MAP = {1: 'Cell', 2: 'dying/dead cells', 3: 'Bead', 4: 'Cluster'}
REVERSE_LABEL_MAP = {value: key for key, value in LABEL_MAP.items()}

In [ ]:
# use color depth = 8 for images shared with annotators; these are already processed images
# we do not do any resizing before cropping the images later into smaller sub-images
# here is the size distribution of the set
# (1600, 2000): 1201, (1944, 2592): 41, (2208, 2758): 69, (2208, 2756): 15

train_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TRAIN_ANNOTS_PATH,
                                color_depth=8, max_larger_side = 5000, max_smaller_side = 5000,
                                normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)

test_dataset = CellMaskDataset(images_path=IMAGES_PATH, annotations_path=TEST_ANNOTS_PATH,
                               color_depth=8, max_larger_side = 5000, max_smaller_side = 5000, 
                               normalize=False, class_names_to_ids_map=REVERSE_LABEL_MAP)

### Output folders and resized image dimensions

In [ ]:
# make sure these folders are generated in advance
OUTPUT_BASE_PATH = os.path.join(os.getcwd(), 'data/cells_cropped')
TRAIN_IMAGE_FOLDER = 'images/train'
TRAIN_MASK_FOLDER = 'masks/train'
TEST_IMAGE_FOLDER = 'images/test'
TEST_MASK_FOLDER = 'masks/test'
# the minimum object mask area to keep the object in the data
MIN_MASK_AREA = 16
# the limit on the larger and smaller sides of the input to Mask R-CNN model
# used for preparing un-cropped training images (whole images)
MASK_RCNN_INPUT_WIDTH = 1200
MASK_RCNN_INPUT_HEIGHT = 800
# the minimum and maximum sizes of sqaure cropped images, used for preparing 
# cropped images (smaller than Mask R-CNN input size specified above) for training
CROPPED_IMAGE_SIZE_MIN = 100
CROPPED_IMAGE_SIZE_MAX = 300
# fixed seed to generate the same sets over multiple runs of the code
# (applicable for cropped imaegd)
SEED = 7
np.random.seed(SEED)

### Generate the train and test data with Mask R-CNN input size resolution
Run the cell below twice, with the train flag set to True and False to generate the train and test data for Mask R-CNN training with un-cropped/full images. This will take the annotated data from the train and test classes and generate cropped sub-images with the annotations, and save them into disk. These are then read for training the model. 

In [ ]:
def prepare_data(train=True):
    
    if train:
        set_desc = 'training'
        image_folder = TRAIN_IMAGE_FOLDER
        mask_folder = TRAIN_MASK_FOLDER
        data_set = train_dataset
    else:
        set_desc = 'testing'
        image_folder = TEST_IMAGE_FOLDER
        mask_folder = TEST_MASK_FOLDER
        data_set = test_dataset

    # keep all the labels in the model label map
    class_ids_of_interest = list(LABEL_MAP.keys())

    num_images = 0
    num_annotations = 0

    # the number of overlapping pixels between crops in each dimension
    # this is larger than the largest expected cell size (in fact, 3 times 
    # more than the expected size of the cells in breadboard images)

    overlap_in_x = 400
    overlap_in_y = 400

    # the step size for the starting point of each crop in x and y dimension
    crop_start_step_x = MASK_RCNN_INPUT_WIDTH - overlap_in_x
    crop_start_step_y = MASK_RCNN_INPUT_HEIGHT - overlap_in_y

    # read the image and the annotations, then parse each
    for idx in range(len(data_set)):

        sample = data_set[idx]
        # image size
        image_height, image_width = sample["image"].shape[:2]

        crop_count = 0
        # overlapping crops
        for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
            for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
                # crop coordinates
                xc_tl = x_start
                yc_tl = y_start
                xc_br = x_start + MASK_RCNN_INPUT_WIDTH
                yc_br = y_start + MASK_RCNN_INPUT_HEIGHT
                # make sure we always crop the image with the given size
                # if we get to the boundaries, extend the crop
                # size inside the image to always get the same size crop
                # this is not really needed, but help with capturing more
                # annotations toward the low/right parts of the image
                if xc_br > image_width:
                    xc_br = image_width 
                    xc_tl = xc_br - MASK_RCNN_INPUT_WIDTH
                if yc_br > image_height:
                    yc_br = image_height
                    yc_tl = yc_br - MASK_RCNN_INPUT_HEIGHT

                crop_coords = [xc_tl, yc_tl, xc_br, yc_br]

                # optimize the crop
                crop_coords = optimize_crop(sample["annotations"], crop_coords, 
                                            overlap_in_x - 1, overlap_in_y - 1, 
                                            image_width, image_height, class_ids_of_interest)

                # crop and block the image
                cropped_sample =  crop_and_block(sample, crop_coords, labels_of_interest=class_ids_of_interest)
                
                # skip empty crops
                if len(cropped_sample['annotations']) < 1:
                    continue

                crop_height, crop_width = cropped_sample["image"].shape[:2]
                # annotation txt file
                crp_mask_name = img_name + '_crp_' + str(crop_count) + '.npy'
                # create the mask annotations
                # the masks are saved as m x crop_height x crop_width array 
                # (m masks with the same resolution as the cropped image)
                # and values as np.uint16 
                # the i-th object in cropped_sample['annotations'] will have 
                # mask values eqaul to (i + 1) in the first possible array index j
                # 0 <= j < m where it will not overlap with any previously processes
                # object
                # in case of having overlapping objects, one of the objects will
                # be reported in the next array index where it does not have 
                # any overlap with objects in that mask

                masks: List[np.array] = [np.zeros((crop_height, crop_width), np.uint16)]
                for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                    # check the area first, make sure to include only large enough objects
                    if current_mask.sum() < MIN_MASK_AREA:
                        continue

                    num_annotations += 1
                    # find the first index with no overlap for the object
                    available_array_index = -1
                    for array_index in range(len(masks)):
                        if masks[array_index][current_mask > 0].sum() == 0:
                            available_array_index = array_index
                            break

                    if available_array_index < 0:
                        # add a new mask
                        masks.append(np.zeros((crop_height, crop_width), np.uint16))
                        available_array_index = len(masks) - 1

                    masks[available_array_index][current_mask > 0] = obj_id + 1

                
                if len(masks) == 0 or masks[0].sum() == 0:
                    # skip this sample if no large enough mask was found
                    continue
                
                # save the cropped image and the annotations for this image, for each image name
                # use _ crop_count
                img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])
                # image jpg file
                crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
                # annotation txt file
                crp_mask_name = img_name + '_crp_' + str(crop_count) + '.npy'
                
                # save the image in jpg format
                cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
                # save the masks array
                np.save(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), np.array(masks))
                # increament the counter for images
                num_images += 1
                # increament the crop counter for this image
                crop_count += 1


    print('Created {} '.format(num_images) + 'images for ' + set_desc)
    print('Created {} '.format(num_annotations) + 'objects for ' + set_desc)

### Generate the train and test cropped images, with the maximum and minimum sizes specified 
Run the cell below twice, with the train flag set to True and False to generate the train and test data for Mask R-CNN training with cropped images. This will take the annotated data from the train and test classes and generate cropped sub-images with the annotations, and save them into disk. These are then read for training the model. 

In [ ]:
def prepare_cropped_data(train=True):
    
    if train:
        set_desc = 'training'
        image_folder = TRAIN_IMAGE_FOLDER
        mask_folder = TRAIN_MASK_FOLDER
        data_set = train_dataset
    else:
        set_desc = 'testing'
        image_folder = TEST_IMAGE_FOLDER
        mask_folder = TEST_MASK_FOLDER
        data_set = test_dataset

    # keep all the labels in the model label map
    class_ids_of_interest = list(LABEL_MAP.keys())

    num_images = 0
    num_annotations = 0
   
    # read the image and the annotations, then parse each
    for idx in range(len(data_set)):

        sample = data_set[idx]
        # image size
        image_height, image_width = sample["image"].shape[:2]
        
        # randomly select a crop size that is used for the whole image
        crop_size: int = int(np.random.rand(1)[0] * (CROPPED_IMAGE_SIZE_MAX - CROPPED_IMAGE_SIZE_MIN) +
                             CROPPED_IMAGE_SIZE_MIN)

        crop_count = 0
        
        # add a slight overlap between the crops
        overlap_in_x = int(0.1 * crop_size)
        overlap_in_y = int(0.1 * crop_size)
        
         # the step size for the starting point of each crop in x and y dimension
        crop_start_step_x = crop_size - overlap_in_x
        crop_start_step_y = crop_size - overlap_in_y
        
        # overlapping crops
        for x_start in range(0, image_width - overlap_in_x, crop_start_step_x):    
            for y_start in range(0, image_height - overlap_in_y, crop_start_step_y):
                # crop coordinates
                xc_tl = x_start
                yc_tl = y_start
                xc_br = x_start + crop_size
                yc_br = y_start + crop_size
                # make sure we always crop the image with the given size
                # if we get to the boundaries, extend the crop
                # size inside the image to always get the same size crop
                # this is not really needed, but help with capturing more
                # annotations toward the low/right parts of the image
                if xc_br > image_width:
                    xc_br = image_width 
                    xc_tl = xc_br - MASK_RCNN_INPUT_WIDTH
                if yc_br > image_height:
                    yc_br = image_height
                    yc_tl = yc_br - MASK_RCNN_INPUT_HEIGHT

                crop_coords = [xc_tl, yc_tl, xc_br, yc_br]

                # optimize the crop
                crop_coords = optimize_crop(sample["annotations"], crop_coords, 
                                            int(crop_size / 2), int(crop_size / 2), 
                                            image_width, image_height, class_ids_of_interest)

                # crop and block the image
                cropped_sample =  crop_and_block(sample, crop_coords, labels_of_interest=class_ids_of_interest)
                
                # skip empty crops
                if len(cropped_sample['annotations']) < 1:
                    continue

                crop_height, crop_width = cropped_sample["image"].shape[:2]
                
                # create the mask annotations
                # the masks are saved as m x crop_height x crop_width array 
                # (m masks with the same resolution as the cropped image)
                # and values as np.uint16 
                # the i-th object in cropped_sample['annotations'] will have 
                # mask values eqaul to (i + 1) in the first possible array index j
                # 0 <= j < m where it will not overlap with any previously processes
                # object
                # in case of having overlapping objects, one of the objects will
                # be reported in the next array index where it does not have 
                # any overlap with objects in that mask

                masks: List[np.array] = [np.zeros((crop_height, crop_width), np.uint16)]
                for obj_id, current_mask in enumerate(cropped_sample["masks"]):
                    
                    # check the area first, make sure to include only large enough objects
                    if current_mask.sum() < MIN_MASK_AREA:
                        continue

                    num_annotations += 1
                    # find the first index with no overlap for the object
                    available_array_index = -1
                    for array_index in range(len(masks)):
                        if masks[array_index][current_mask > 0].sum() == 0:
                            available_array_index = array_index
                            break

                    if available_array_index < 0:
                        # add a new mask
                        masks.append(np.zeros((crop_height, crop_width), np.uint16))
                        available_array_index = len(masks) - 1

                    masks[available_array_index][current_mask > 0] = obj_id + 1
                    
                if len(masks) == 0 or masks[0].sum() == 0:
                    # skip this sample if no large enough mask was found
                    continue
                
                # save the cropped image and the annotations for this image, for each image name
                # use _ crop_count
                img_name = ".".join(cropped_sample["name"].strip().split('.')[:-1])
                # image jpg file
                crp_img_name = img_name + '_crp_' + str(crop_count) + '.jpg'
                # annotation txt file
                crp_mask_name = img_name + '_crp_' + str(crop_count) + '.npy'
                
                # save the image in jpg format
                cv2.imwrite(os.path.join(OUTPUT_BASE_PATH, image_folder, crp_img_name), cropped_sample["image"])
                # save the masks array
                np.save(os.path.join(OUTPUT_BASE_PATH, mask_folder, crp_mask_name), np.array(masks))
                # increament the counter for images
                num_images += 1
                # increament the crop counter for this image
                crop_count += 1


    print('Created {} '.format(num_images) + 'images for ' + set_desc)
    print('Created {} '.format(num_annotations) + 'objects for ' + set_desc)

In [ ]:
prepare_data(True)

In [ ]:
prepare_data(False)

In [ ]:
prepare_cropped_data(train=False)

In [ ]:
prepare_cropped_data(train=True)